# Loading

In [1]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "data/jobs.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

5


In [3]:
type(docs[0])

langchain_core.documents.base.Document

In [2]:
docs[0].page_content
docs[0].metadata

{'producer': 'Prince 16.1 (www.princexml.com)',
 'creator': 'PyPDF',
 'creationdate': '',
 'title': 'jobs',
 'source': 'data/jobs.pdf',
 'total_pages': 5,
 'page': 0,
 'page_label': '1'}

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
def load_pdf(pdf_path: str) -> list[Document]:
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

In [4]:
documents=load_pdf(file_path)

In [5]:
len(documents[0].page_content)


1688

# Chunking

In [6]:
# token split PDF
from langchain_text_splitters import TokenTextSplitter
token_splitter = TokenTextSplitter(chunk_size=50, chunk_overlap=20)
split_docs = token_splitter.split_documents(documents)
print(len(split_docs))
print("Text: ", split_docs[0].page_content)
print("Metadata: ", split_docs[0].metadata)



192
Text:  Danh sách JD của RikkeiSoft
• 1. Frontend Developer (React/Next.js)
◦ Mô tả công việc:
�
Metadata:  {'producer': 'Prince 16.1 (www.princexml.com)', 'creator': 'PyPDF', 'creationdate': '', 'title': 'jobs', 'source': 'data/jobs.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


In [7]:
# markdown split

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
from langchain_text_splitters import MarkdownHeaderTextSplitter
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=["#"])
split_docs = markdown_splitter.split_text(documents[0].page_content)

ValueError: not enough values to unpack (expected 2, got 1)

In [8]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
split_docs = text_splitter.split_documents(documents)
print(len(split_docs))
print("Content: ", split_docs[0].page_content)
print("Metadata: ", split_docs[0].metadata)



38
Content:  Danh sách JD của RikkeiSoft
• 1. Frontend Developer (React/Next.js)
◦ Mô tả công việc:
▪ Phát triển giao diện web hiện đại, tối ưu trải nghiệm người dùng.
▪ Xây dựng các component tái sử dụng bằng React/Next.js.
▪ Tối ưu hiệu năng client-side, SEO và tốc độ tải trang.
Metadata:  {'producer': 'Prince 16.1 (www.princexml.com)', 'creator': 'PyPDF', 'creationdate': '', 'title': 'jobs', 'source': 'data/jobs.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


In [19]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
def load_and_split(path: str):
    loader = PyPDFLoader(path)
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )
    split_docs = splitter.split_documents(documents)
    return split_docs

# Embeddings

In [9]:
from openai import OpenAI
from config import settings
client = OpenAI(api_key=settings.LLM_API_KEY, base_url=settings.LLM_BASE_URL)
def embed_query(query: str):
    response = client.embeddings.create(input=query, model=settings.LLM_EMBEDDING_MODEL)
    return response.data[0].embedding


In [10]:
embed_query("Hello, world!")


[-0.020920176059007645,
 0.009755219332873821,
 0.004780902992933989,
 -0.05942125245928764,
 0.005082839634269476,
 0.0008645110065117478,
 -0.004584700334817171,
 0.005077085457742214,
 0.01929636485874653,
 0.01731793023645878,
 -0.008134705945849419,
 -0.009040719829499722,
 -0.004410189110785723,
 0.030996358022093773,
 0.10537973791360855,
 0.010174652561545372,
 0.023028461262583733,
 -0.02839803323149681,
 0.0023683798499405384,
 -0.006660488899797201,
 0.0028257493395358324,
 -0.001545771025121212,
 0.03322514519095421,
 -0.008438869379460812,
 0.0032804706133902073,
 0.007675922010093927,
 0.03673441335558891,
 -0.016130663454532623,
 0.02533765323460102,
 0.021699391305446625,
 -0.0008153883391059935,
 0.015893541276454926,
 -0.0450279638171196,
 0.0075905160047113895,
 9.405229502590373e-05,
 0.03321804851293564,
 -0.0026741567999124527,
 -0.014441574923694134,
 -0.004091567825525999,
 0.00832407083362341,
 -0.012123116292059422,
 -0.0029273992404341698,
 0.0070877987891435

In [11]:
vector_1 = embed_query("Rikkesoft là gì?")
vector_2=embed_query("Rikkeisoft là công ty công nghệ!")
# Cosine similarity



In [12]:
vector_3 = embed_query("Hôm nay trời mưa")

In [13]:
import numpy as np

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))


In [14]:
cosine_similarity(vector_1, vector_2)

np.float64(0.8053668932502924)

In [15]:
cosine_similarity(vector_1, vector_3)

np.float64(0.5479871498053407)

# Vector Database

In [16]:
from typing import List, Optional

from langchain_chroma import Chroma
from langchain_core.documents import Document
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
from src.embeddings import OpenAIEmbeddingFunction
from config import settings
from langchain_core.documents import Document
embeddings = OpenAIEmbeddingFunction(
    # api_key=settings.LLM_API_KEY,
    model=settings.LLM_EMBEDDING_MODEL,
)
def create_or_get_vector_store(
    collection_name: str = "documents",
    persist_directory: str = "vector_store",
    delete_existing: bool = False,
):
    if delete_existing:
        Chroma(
            persist_directory=persist_directory,
            collection_name=collection_name,
            embedding_function=embeddings,
        ).delete_collection(collection_name)
    return Chroma(
        persist_directory=persist_directory,
        collection_name=collection_name,
        embedding_function=embeddings,
    )


In [17]:
def ingest_documents(
    collection_name: str = "documents",
    persist_directory: str = "vector_store",
    documents: Optional[List[Document]] = None,
):
    vector_store = create_or_get_vector_store(collection_name, persist_directory)
    vector_store.add_documents(documents)
    return vector_store

In [20]:
documents = load_and_split("data/jobs.pdf")
print(f"Ingesting {len(documents)} documents from data/jobs.pdf")
ingest_documents(documents=documents)
print(f"Ingested {len(documents)} documents from data/jobs.pdf")

Ingesting 21 documents from data/jobs.pdf
Ingested 21 documents from data/jobs.pdf


# Retriever

In [21]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from typing import List
def get_retriever(query: str)-> List[Document]:
    vectorstore = create_or_get_vector_store()
    return vectorstore.similarity_search(query, k=3)

In [22]:
retriever = get_retriever("QA Engineer (Manual/Automation")
retriever[0].page_content

'▪ Biết sử dụng các công cụ CI/CD (GitHub Actions/GitLab CI/Jenkins...).\n▪ Hiểu về networking cơ bản, bảo mật hệ thống, Linux.\n▪ Kỹ năng scripting (Bash/Python) tốt.\n• 4. QA Engineer (Manual/Automation)\n◦ Mô tả công việc:\n▪ Xây dựng test plan, test case cho các tính năng mới.\n▪ Thực hiện test manual (functional, regression, UI/UX…).\n▪ Viết và duy trì test automation (API/UI) khi cần thiết.\n▪ Phối hợp với team Dev/PM để tái hiện và theo dõi bug.'

# RAG PIPELINE

In [23]:
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from typing import List, Tuple
from pydantic import BaseModel, Field
from config import settings
from src.retriever import get_retriever
from prompt.prompt import RAG_SYSTEM_PROMPT
# Output structure definition
class Source(BaseModel):
    """Source information"""
    document: int = Field(description="Document number")
    source: str = Field(description="Source file path or link")
    page: int = Field(description="Page number in the source document", default=None)
class RAGResponse(BaseModel):
    """Structured output for RAG responses"""
    answer: str = Field(description="The answer to the question based on context documents")
    sources: List[Source] = Field(description="List of sources with document number, source path, and page number")


def _format_documents(docs: List[Document]) -> str:
    """Format retrieved documents into JSON string format"""
    documents_list = []
    for i, doc in enumerate(docs, 1):
        doc_dict = {
            "document": i,
            "source": doc.metadata.get("source", "Unknown"),
            "page": doc.metadata.get("page", None),
            "content": doc.page_content.strip()
        }
        documents_list.append(doc_dict)
    
    return json.dumps(documents_list, ensure_ascii=False, indent=2)


# Create output parser
output_parser = PydanticOutputParser(pydantic_object=RAGResponse)

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
    ("human", "{question}")
])


def chat_with_rag(query: str) -> Tuple[RAGResponse, List[Document]]:
    llm = ChatOpenAI(
        api_key=settings.LLM_API_KEY,
        model=settings.LLM_CHAT_MODEL,
        base_url=settings.LLM_BASE_URL,
        temperature=0.1
    )
    
    docs = get_retriever(query)
    context = _format_documents(docs)
    
    # Chain with structured output parser
    chain = RAG_PROMPT | llm | output_parser
    
    result = chain.invoke({"question": query, "context": context})
    return result, docs


In [24]:
chat_with_rag("Trong công ty có những vị trí gì")

(RAGResponse(answer='Dựa trên các tài liệu được cung cấp, công ty có các vị trí sau:\n1. Product Manager (vị trí số 5)\n2. Mobile Developer (iOS/Android/Flutter) (vị trí số 9)\n3. Technical Lead / Software Architect (vị trí số 10)\nNgoài ra, tài liệu còn đề cập đến các yêu cầu chuyên môn cho một vị trí liên quan đến Machine Learning/Data Science (như nắm vững thuật toán ML, Python, Pandas, TensorFlow/PyTorch) nhưng tên vị trí cụ thể không được liệt kê rõ ràng trong đoạn trích.', sources=[Source(document=1, source='data/jobs.pdf', page=3), Source(document=2, source='data/jobs.pdf', page=1), Source(document=3, source='data/jobs.pdf', page=3)]),
 [Document(id='c9b233bd-4909-4906-9265-f4d84060ab29', metadata={'title': 'jobs', 'page': 3, 'source': 'data/jobs.pdf', 'producer': 'Prince 16.1 (www.princexml.com)', 'creator': 'PyPDF', 'page_label': '4', 'total_pages': 5, 'creationdate': ''}, page_content='đương.\n▪ Nắm vững thống kê cơ bản và các thuật toán ML phổ biến.\n▪ Có kinh nghiệm với Pyt